# บทที่ 08 - รูปแบบการออกแบบหลายตัวแทน


## การตั้งค่า


In [ ]:
import logging
logging.getLogger("agent_framework.foundry").setLevel(logging.ERROR)

%pip install agent-framework azure-ai-projects azure-identity python-dotenv --quiet

import os
import asyncio
import dotenv

from agent_framework import AgentResponseUpdate, WorkflowBuilder
from agent_framework.foundry import FoundryChatClient
from azure.identity import DefaultAzureCredential

dotenv.load_dotenv()

endpoint = os.getenv("AZURE_AI_PROJECT_ENDPOINT")
deployment_name = os.getenv("AZURE_AI_MODEL_DEPLOYMENT_NAME")

missing = [k for k, v in {
    "AZURE_AI_PROJECT_ENDPOINT": endpoint,
    "AZURE_AI_MODEL_DEPLOYMENT_NAME": deployment_name
}.items() if not v]

if missing:
    raise ValueError(
        f"Missing required environment variables: {', '.join(missing)}. "
        "Please set them as environment variables (e.g., in your .env file or shell environment)."
    )

In [ ]:
# Create the Microsoft Foundry client
client = FoundryChatClient(
    project_endpoint=endpoint,
    model=deployment_name,
    credential=DefaultAzureCredential()
)

## ทำไมต้องใช้ระบบหลายตัวแทน?

งานในโลกแห่งความเป็นจริง เช่น การวางแผนการเดินทาง ต้องใช้ความเชี่ยวชาญในหลายด้าน — โลจิสติกส์ ความรู้ท้องถิ่น งบประมาณ และอื่น ๆ ตัวแทนเพียงคนเดียวที่พยายามจัดการทุกอย่างจะกลายเป็นเรื่องยุ่งยากอย่างรวดเร็ว

ระบบหลายตัวแทนแก้ปัญหานี้ด้วย **ความเชี่ยวชาญเฉพาะทาง**: แต่ละตัวแทนจะเน้นที่ความเชี่ยวชาญด้านใดด้านหนึ่ง ทำให้ได้ผลลัพธ์ที่มีคุณภาพสูงกว่าผู้เชี่ยวชาญทั่วไป นอกจากนี้ยังช่วยเพิ่ม **ความสามารถในการขยายตัว** — คุณสามารถเพิ่มตัวแทนใหม่ (เช่น ผู้เชี่ยวชาญด้านเที่ยวบิน นักวิจารณ์ร้านอาหาร) ได้โดยไม่ต้องเขียนโฟลว์งานที่มีอยู่ใหม่ ตัวแทนเหล่านี้จะทำงานร่วมกันผ่านกระบวนการที่มีโครงสร้าง โดยส่งผ่านบริบทจากตัวแทนหนึ่งไปยังอีกตัวแทนหนึ่ง


## การสร้างเอเย่นต์เฉพาะทาง


In [ ]:
planner_agent = client.as_agent(
    name="TravelPlanner",
    instructions="You are a travel planning specialist. Create detailed trip itineraries based on the traveler's preferences. Include daily schedules, must-see attractions, and logistical tips.",
)

concierge_agent = client.as_agent(
    name="TravelConcierge",
    instructions="You are a travel concierge who reviews and enhances trip plans. Review the plan for completeness, add local insider tips, suggest restaurants, and identify potential issues. Provide your feedback in a constructive format.",
)

## สร้างกระบวนการทำงานแบบลำดับ

`WorkflowBuilder` ช่วยให้คุณเชื่อมต่อเอเจนต์เข้ากับกราฟที่มีทิศทางได้ ที่นี่เราจะสร้างสายงานแบบสองขั้นตอนง่ายๆ: **TravelPlanner** ร่างแผนการเดินทาง แล้ว **TravelConcierge** จะตรวจทานและปรับปรุงแผนดังกล่าว


In [ ]:
workflow = WorkflowBuilder(start_executor=planner_agent) \
    .add_edge(planner_agent, concierge_agent) \
    .build()

last_author = None
events = workflow.run("Plan a 5-day trip to Paris for a food-loving couple on a $3000 budget.", stream=True)
async for event in events:
    if event.type == "output" and isinstance(event.data, AgentResponseUpdate):
        update = event.data
        author = update.author_name
        if author != last_author:
            if last_author is not None:
                print()
            print(f"\n{'='*50}")
            print(f"🤖 {author}:")
            print(f"{'='*50}")
            last_author = author
        print(update.text, end="", flush=True)

## การเพิ่มตัวแทนเพิ่มเติมในขั้นตอนการทำงาน

หนึ่งในข้อได้เปรียบที่ใหญ่ที่สุดของรูปแบบตัวแทนหลายตัวคือความง่ายในการขยาย ด้านล่างนี้เราจะเพิ่มตัวแทน **BudgetReviewer** ที่ตรวจสอบแผนกับงบประมาณของนักเดินทาง ทำเครื่องหมายรายการที่อาจทำให้ค่าใช้จ่ายเกินขีดจำกัด และเสนอทางเลือกที่ช่วยประหยัดเงิน ขั้นตอนการทำงานตอนนี้ดำเนินการผ่านตัวแทนสามตัวตามลำดับ:

```
TravelPlanner → TravelConcierge → BudgetReviewer
```


In [ ]:
budget_agent = client.as_agent(
    name="BudgetReviewer",
    instructions="You are a budget-conscious travel advisor. Review the proposed trip plan and concierge enhancements against the traveler's stated budget. Estimate costs for flights, hotels, meals, and activities. Flag anything that risks exceeding the budget and suggest cost-saving alternatives while preserving the trip's quality.",
)

extended_workflow = WorkflowBuilder(start_executor=planner_agent) \
    .add_edge(planner_agent, concierge_agent) \
    .add_edge(concierge_agent, budget_agent) \
    .build()

last_author = None
events = extended_workflow.run("Plan a 5-day trip to Paris for a food-loving couple on a $3000 budget.", stream=True)
async for event in events:
    if event.type == "output" and isinstance(event.data, AgentResponseUpdate):
        update = event.data
        author = update.author_name
        if author != last_author:
            if last_author is not None:
                print()
            print(f"\n{'='*50}")
            print(f"🤖 {author}:")
            print(f"{'='*50}")
            last_author = author
        print(update.text, end="", flush=True)

## สรุป

ในบทเรียนนี้คุณได้เรียนรู้วิธีการ:

1. **สร้างเอเย่นต์เฉพาะทาง** — แต่ละตัวมีบทบาทเจาะจง (วางแผน, คอนเซียร์จ, ตรวจสอบงบประมาณ)
2. **เชื่อมต่อเอเย่นต์ในลำดับขั้นตอนการทำงาน** ด้วย `WorkflowBuilder` และ `add_edge`
3. **สตรีมผลลัพธ์** จากสายงานหลายเอเย่นต์ พร้อมติดตามว่าเอเย่นต์ใดกำลังพูด
4. **ขยายลำดับขั้นตอนการทำงาน** โดยเพิ่มเอเย่นต์ใหม่ในห่วงโซ่โดยไม่ต้องแก้ไขเอเย่นต์ที่มีอยู่

รูปแบบการออกแบบหลายเอเย่นต์ช่วยให้แต่ละเอเย่นต์ทำงานอย่างเรียบง่าย ในขณะที่ผลิตผลลัพธ์ที่ลึกซึ้งกว่าและผ่านการตรวจสอบอย่างละเอียดกว่าการใช้เอเย่นต์เพียงตัวเดียวได้


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**ปฏิเสธความรับผิดชอบ**:
เอกสารนี้ได้รับการแปลโดยใช้บริการแปลภาษา AI [Co-op Translator](https://github.com/Azure/co-op-translator) ขณะที่เราพยายามให้ความถูกต้อง โปรดทราบว่าการแปลโดยอัตโนมัติอาจมีข้อผิดพลาดหรือความไม่ถูกต้อง เอกสารต้นฉบับในภาษาต้นทางควรถูกพิจารณาเป็นแหล่งข้อมูลที่เชื่อถือได้ สำหรับข้อมูลที่สำคัญ แนะนำให้ใช้การแปลโดยมนุษย์มืออาชีพ เราไม่รับผิดชอบต่อความเข้าใจผิดหรือการตีความที่ผิดพลาดที่เกิดขึ้นจากการใช้การแปลนี้
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
